In [1]:

import torch
from rich import print
from datasets import  final_extinctionrisk_dataframe, final_extinctionrisk_noth_dataframe
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from pytorch_tabular import TabularModel
from pytorch_tabular.models import TabNetModelConfig
from pytorch_tabular.config import (
    DataConfig,
    OptimizerConfig,
    TrainerConfig,
)

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device == "cuda":
    torch.set_float32_matmul_precision('high')

# MODEL 1

In [3]:
model, data = final_extinctionrisk_dataframe()
categorical_data = data.drop(model.numeric, axis=1)
cat_col_names = list(categorical_data.columns)
num_col_names = model.numeric

print(f"Data Shape: {data.shape} | # of cat cols: {len(cat_col_names)} | # of num cols: {len(num_col_names)}")
print(f"[bold dodger_blue2] Features: {num_col_names + cat_col_names}[/bold dodger_blue2]")
print(f"[bold purple4]Target: {model.label}[/bold purple4]")

Data Shape: (9053, 32) | # of cat cols: 14 | # of num cols: 18

 Features: ['Beak_length_culmen', 'Beak_depth', 'Tarsus_length', 'Wing_length', 'Hand_wing_index', 'Tail_length', 
'Minimum_latitude', 'Maximum_latitude', 'Minimum_elevation', 'Elevational_range', 'Maximum_elevation', 
'Habitat_breadth', 'Diet_breadth', 'Adult_survival_annual', 'Generation_length', 'Range_size', 'Body_mass', 
'Clutch_size', 'Order', 'Family', 'Agriculture', 'Hunting', 'Invasive_species', 'Climate_change', 
'Primary_lifestyle', 'Island_restricted_breeding', 'Latitudinal_range', 'Realm', 'Diet', 'Habitat', 'Migration', 
'Extinction_risk']

Target: Extinction_risk

In [4]:

train, test = train_test_split(data, random_state=42, test_size=0.2)
val = test
#train, val = train_test_split(train, random_state=42, test_size=0.2)
print(f"Train Shape: {train.shape} | Val Shape: {val.shape} | Test Shape: {test.shape}")

Train Shape: (7242, 32) | Val Shape: (1811, 32) | Test Shape: (1811, 32)

In [5]:
data_config = DataConfig(
    target=[
        model.label
    ],  
    continuous_cols=num_col_names,
    categorical_cols=cat_col_names,
    num_workers = 0, 
    pin_memory=True
)
trainer_config = TrainerConfig(
    batch_size=128,
    max_epochs=20,
)
optimizer_config = OptimizerConfig()

model_config = TabNetModelConfig(
    n_d = 8,
    n_a = 8,
    n_steps = 3,
    gamma = 1.3,
    n_independent = 2,
    n_shared = 2,
    virtual_batch_size=128,
    mask_type = 'sparsemax',
    task="classification",
    head = 'LinearHead',
    embedding_dropout = 0.0,
    batch_norm_continuous_input = True,
    learning_rate = 0.001,
    seed = 42,
    metrics = ["accuracy"]
)

In [6]:
tabular_model = TabularModel(
    data_config=data_config,
    model_config=model_config,
    optimizer_config=optimizer_config,
    trainer_config=trainer_config,
    verbose=True
)

2026-03-11 15:45:18,064 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off


In [7]:
tabular_model.fit(train=train, validation=val)

Seed set to 42
2026-03-11 15:45:18,142 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-03-11 15:45:18,161 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for classification task
c:\Users\anmarch\.conda\envs\pytorch_tabular\lib\site-packages\pytorch_tabular\tabular_datamodule.py:388: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.52748677  0.82480189 -0.52748677 ... -0.52748677  0.6745476
 -0.52748677]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data.loc[:, self.config.continuous_cols] = self.scaler.fit_transform(
c:\Users\anmarch\.conda\envs\pytorch_tabular\lib\site-packages\pytorch_tabular\tabular_datamodule.py:388: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-1.53845304 -0.33916505  2.56143847 ...  1.83628759  1.66894508
  1.7805067

┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _embedding_layer │ Identity         │      0 │ train │     0 │
│ 1 │ _backbone        │ TabNetBackbone   │ 26.2 K │ train │     0 │
│ 2 │ _head            │ Identity         │      0 │ train │     0 │
│ 3 │ loss             │ CrossEntropyLoss │      0 │ train │     0 │
└───┴──────────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 26.2 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 26.2 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 122                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

c:\Users\anmarch\.conda\envs\pytorch_tabular\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.
py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of
the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.

c:\Users\anmarch\.conda\envs\pytorch_tabular\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.
py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value 
of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=20` reached.


2026-03-11 15:46:02,111 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-03-11 15:46:02,111 - {pytorch_tabular.tabular_model:1531} - INFO - Loading the best model
c:\Users\anmarch\.conda\envs\pytorch_tabular\lib\site-packages\pytorch_tabular\utils\python_utils.py:92: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `wei

In [8]:
pred_df = tabular_model.predict(test)["Extinction_risk_prediction"].to_numpy()

mapping = {'Lower_risk':2, 'Higher_risk':1}
test_y = test[model.label]
test_y = test_y.map(mapping).to_numpy()

c:\Users\anmarch\.conda\envs\pytorch_tabular\lib\site-packages\pytorch_tabular\tabular_datamodule.py:392: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.52748677 -0.52748677 -0.52748677 ... -0.52748677 -0.52748677
 -0.52748677]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data.loc[:, self.config.continuous_cols] = self.scaler.transform(data.loc[:, self.config.continuous_cols])
c:\Users\anmarch\.conda\envs\pytorch_tabular\lib\site-packages\pytorch_tabular\tabular_datamodule.py:392: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.45072672 -0.45072672 -1.00853509 ...  2.89612349 -0.45072672
 -1.45478179]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data.loc[:, self.config.continuous_cols] = self.scaler.transform(data.loc[:, self.config.continuous_cols])


# With Human Threats Included

In [9]:
f1 = f1_score(test_y, pred_df, average=None)
precision = precision_score(test_y, pred_df, average=None)
recall = recall_score(test_y, pred_df, average=None)

print("F1:"+str(f1)+", Precision:"+str(precision)+", Recall:"+str(recall))

F1:[0.96866097 0.99246575], Precision:[0.97421203 0.99110807], Recall:[0.9631728  0.99382716]

# MODEL 2

In [10]:
model, data = final_extinctionrisk_noth_dataframe()
categorical_data = data.drop(model.numeric, axis=1)
cat_col_names = list(categorical_data.columns)
num_col_names = model.numeric

train, test = train_test_split(data, random_state=42, test_size=0.2)
val = test
#train, val = train_test_split(train, random_state=42, test_size=0.2)

data_config = DataConfig(
    target=[
        model.label
    ],  
    continuous_cols=num_col_names,
    categorical_cols=cat_col_names,
    num_workers = 0, 
    pin_memory=True
)
trainer_config = TrainerConfig(
    batch_size=128,
    max_epochs=20,
)
optimizer_config = OptimizerConfig()

model_config = TabNetModelConfig(
    n_d = 8,
    n_a = 8,
    n_steps = 3,
    gamma = 1.3,
    n_independent = 2,
    n_shared = 2,
    virtual_batch_size=128,
    mask_type = 'sparsemax',
    task="classification",
    head = 'LinearHead',
    embedding_dropout = 0.0,
    batch_norm_continuous_input = True,
    learning_rate = 0.001,
    seed = 42,
    metrics = ["accuracy"]
)

tabular_model = TabularModel(
    data_config=data_config,
    model_config=model_config,
    optimizer_config=optimizer_config,
    trainer_config=trainer_config,
    verbose=True
)

tabular_model.fit(train=train, validation=val)
pred_df = tabular_model.predict(test)["Extinction_risk_prediction"].to_numpy()

mapping = {'Lower_risk':2, 'Higher_risk':1}
test_y = test[model.label]
test_y = test_y.map(mapping).to_numpy()


2026-03-11 15:46:02,367 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off
Seed set to 42
2026-03-11 15:46:02,391 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-03-11 15:46:02,395 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for classification task
c:\Users\anmarch\.conda\envs\pytorch_tabular\lib\site-packages\pytorch_tabular\tabular_datamodule.py:388: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.52748677  0.82480189 -0.52748677 ... -0.52748677  0.6745476
 -0.52748677]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data.loc[:, self.config.continuous_cols] = self.scaler.fit_transform(
c:\Users\anmarch\.conda\envs\pytorch_tabular\lib\site-packages\pytorch_tabular\tabular_datamodule.py:388: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a fu

┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _embedding_layer │ Identity         │      0 │ train │     0 │
│ 1 │ _backbone        │ TabNetBackbone   │ 25.8 K │ train │     0 │
│ 2 │ _head            │ Identity         │      0 │ train │     0 │
│ 3 │ loss             │ CrossEntropyLoss │      0 │ train │     0 │
└───┴──────────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 25.8 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 25.8 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 118                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

c:\Users\anmarch\.conda\envs\pytorch_tabular\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.
py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of
the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.

c:\Users\anmarch\.conda\envs\pytorch_tabular\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.
py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value 
of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=20` reached.


2026-03-11 15:46:49,034 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-03-11 15:46:49,034 - {pytorch_tabular.tabular_model:1531} - INFO - Loading the best model
c:\Users\anmarch\.conda\envs\pytorch_tabular\lib\site-packages\pytorch_tabular\utils\python_utils.py:92: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `wei

# Without Human Threats

In [12]:
f1 = f1_score(test_y, pred_df, average=None)
precision = precision_score(test_y, pred_df, average=None)
recall = recall_score(test_y, pred_df, average=None)

print("F1:"+str(f1)+", Precision:"+str(precision)+", Recall:"+str(recall))

F1:[0.99430199 0.99863014], Precision:[1.         0.99726402], Recall:[0.98866856 1.        ]